# Colab — activation extraction & probing (boolean flags)

**Colab:** Select Kernel → **Colab** → **GPU**, then run top-to-bottom.

**Local (Mac):** Select the project `.venv` kernel, run `uv sync` once. Setup skips pip and uses CPU for probing only (no extraction).

1. Push latest code to GitHub before Colab runs.
2. Configuration: `LANGUAGE = "java"`, upload `java_train.jsonl` to Drive.
3. Extraction on Colab GPU; probing works locally on saved activations.

In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

REQUIRE_CUDA = True
WARN_UNLESS_SUBSTR = ("A100", "H100", "L4", "G4", "T4")
DEFAULT_GIT_URL = "https://github.com/nolanlwin/mech-interp-coding-llms.git"
CLONE_DIR = Path("/content/mech-interp-coding-llms")

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")


def _has_scripts(root: Path) -> bool:
    return (root / "scripts" / "activation_pipeline.py").is_file()


def find_repo_root() -> Path | None:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        CLONE_DIR,
        Path("/content/drive/MyDrive/mech-interp-coding-llms"),
        Path("/content/drive/MyDrive/mech-interp/mech-interp-coding-llms"),
    ]
    if IN_COLAB:
        drive_root = Path("/content/drive/MyDrive")
        if drive_root.is_dir():
            for hit in drive_root.rglob("activation_pipeline.py"):
                if hit.parent.name == "scripts" and _has_scripts(hit.parent.parent):
                    return hit.parent.parent.resolve()
    for c in candidates:
        if _has_scripts(c):
            return c.resolve()
    return None


FORCE_RECLONE = False  # set True once to wipe /content clone and re-fetch from GitHub

def _has_java_scripts(root: Path) -> bool:
    return (root / "scripts" / "java_variable_occurrences.py").is_file()


def ensure_repo_on_colab() -> Path:
    """On Colab, always use CLONE_DIR and git pull so scripts match GitHub main."""
    if not IN_COLAB:
        found = find_repo_root()
        if found is not None:
            return found
        raise FileNotFoundError(
            "Run this notebook from the project root, or install scripts/ locally."
        )

    import shutil

    git_url = os.environ.get("REPO_GIT_URL", DEFAULT_GIT_URL)
    if FORCE_RECLONE and CLONE_DIR.is_dir():
        shutil.rmtree(CLONE_DIR)

    if CLONE_DIR.is_dir() and (CLONE_DIR / ".git").is_dir():
        print(f"git pull → {CLONE_DIR}")
        subprocess.check_call(["git", "-C", str(CLONE_DIR), "pull", "--ff-only"])
    else:
        if CLONE_DIR.is_dir():
            shutil.rmtree(CLONE_DIR)
        print(f"Cloning {git_url} → {CLONE_DIR}")
        subprocess.check_call(["git", "clone", "--depth", "1", git_url, str(CLONE_DIR)])

    if not _has_scripts(CLONE_DIR):
        raise FileNotFoundError(f"Clone succeeded but scripts/ missing under {CLONE_DIR}")
    return CLONE_DIR.resolve()


def require_language_support(language: str) -> None:
    if language == "python":
        return
    if _has_java_scripts(REPO_ROOT):
        return
    raise RuntimeError(
        "Java scripts are not in the cloned repo yet.\n"
        "  1. On your laptop: commit + push Java changes to GitHub main.\n"
        "  2. In Colab: set FORCE_RECLONE = True, re-run setup, then Configuration.\n"
        f"  REPO_ROOT={REPO_ROOT}\n"
        "  Missing: scripts/java_variable_occurrences.py"
    )


REQUIRE_CUDA = IN_COLAB  # local Mac: CPU probing only; Colab: need GPU for extraction

def _ensure_deps() -> None:
    if IN_COLAB:
        subprocess.check_call(
            [
                sys.executable, "-m", "pip", "install", "-q",
                "torch", "transformers", "accelerate", "tqdm", "numpy",
                "scikit-learn", "matplotlib", "tree-sitter", "tree-sitter-java",
            ]
        )
        return
    # Local: use project venv (uv sync). No pip in uv venvs.
    missing = []
    for mod in ("torch", "transformers", "tqdm", "numpy", "sklearn", "matplotlib", "tree_sitter", "tree_sitter_java"):
        try:
            __import__(mod)
        except ImportError:
            missing.append(mod)
    if missing:
        raise RuntimeError(
            "Missing packages in local kernel: " + ", ".join(missing) + "\n"
            "From repo root run:  uv sync\n"
            "Then select the .venv kernel for this notebook."
        )
    print("Local mode: using existing venv (run `uv sync` if imports fail).")


REPO_ROOT = ensure_repo_on_colab()
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "scripts"))

_ensure_deps()

import torch


def check_gpu() -> str:
    if not torch.cuda.is_available():
        if REQUIRE_CUDA:
            raise RuntimeError(
                "No CUDA GPU. Select Kernel → Colab → New Colab Server → GPU."
            )
        return "cpu"
    name = torch.cuda.get_device_name(0)
    upper = name.upper()
    if not any(tag in upper for tag in WARN_UNLESS_SUBSTR):
        print(f"WARNING: uncommon GPU {name!r} — extraction should still work.")
    return name


gpu_name = check_gpu()
print(f"IN_COLAB={IN_COLAB}")
print(f"REPO_ROOT={REPO_ROOT}")
print(f"GPU: {gpu_name}")
if IN_COLAB and gpu_name != "cpu":
    print("OK to proceed — any Colab GPU is fine for Qwen2.5-1.5B extraction.")

/Users/naingoolwin/Desktop/mech-interp-coding-llms/.venv/bin/python: No module named pip


CalledProcessError: Command '['/Users/naingoolwin/Desktop/mech-interp-coding-llms/.venv/bin/python', '-m', 'pip', 'install', '-q', 'torch', 'transformers', 'accelerate', 'tqdm', 'numpy', 'scikit-learn', 'matplotlib', 'tree-sitter', 'tree-sitter-java']' returned non-zero exit status 1.

## Configuration

Set **`LANGUAGE`**, upload JSONL to `My Drive/mech-interp/jsonl/<language>_train.jsonl`.

**You do not need the full train split for probing.** Default `EXTRACTION_MODE = "probe"` stops after ~25k input rows or ~12k `.npz` files — enough for 8k probe samples in ~2 hours, not 8+ hours for 454k rows.

| Language | Drive path |
|----------|------------|
| java | `My Drive/mech-interp/jsonl/java_train.jsonl` |
| python | `My Drive/mech-interp/jsonl/python_train.jsonl` |

Set `EXTRACTION_MODE = "full"` only if you intentionally want the entire split.

In [12]:
# --- change this for Java vs Python ---
LANGUAGE = "java"  # "python" | "java"

TRAIN_FILENAME = f"{LANGUAGE}_train.jsonl"

if IN_COLAB:
    DRIVE_ROOT = Path("/content/drive/MyDrive/mech-interp")
else:
    DRIVE_ROOT = REPO_ROOT / "outputs"  # local smoke test

# Keep legacy python path on Drive (activations/); java gets its own folder.
if LANGUAGE == "python":
    ACTIVATIONS_DIR = DRIVE_ROOT / "activations"
else:
    ACTIVATIONS_DIR = DRIVE_ROOT / f"activations_{LANGUAGE}"

MANIFEST = ACTIVATIONS_DIR / "manifest.jsonl"
NPZ_DIR = ACTIVATIONS_DIR / "npz"
OCCURRENCES_JSONL = ACTIVATIONS_DIR / f"{LANGUAGE}_boolean_flag_occurrences.jsonl"
CANONICAL_INPUT = DRIVE_ROOT / "jsonl" / TRAIN_FILENAME

MODEL_ID = "Qwen/Qwen2.5-1.5B"

# --- extraction scope (probe vs full train) ---
EXTRACTION_MODE = "probe"       # "probe" | "full"
TARGET_COMPLETE_ROWS = 25_000   # probe: input rows to process (not npz count)
TARGET_NPZ = 12_000             # probe: stop early when this many .npz exist
PROBE_MAX_SAMPLES = 8_000       # probing uses at most this many activations

CHUNK_ROWS = 500                # smaller = less GPU memory, stops sooner
MAX_CHUNKS = 1                  # one subprocess per run; re-run cell to continue
STOP_FILE = ACTIVATIONS_DIR / "STOP_EXTRACTION"  # create on Drive to stop mid-chunk
TOTAL_INPUT_ROWS = 454452       # full train size (display only)
RESUME = True
POOLING = "mean"
EXTRACT_OCCURRENCES = False

ACTIVATIONS_DIR.mkdir(parents=True, exist_ok=True)
NPZ_DIR.mkdir(parents=True, exist_ok=True)


def resolve_input_jsonl(language: str) -> Path:
    """Find <language>_train.jsonl locally or anywhere on mounted Drive."""
    fname = f"{language}_train.jsonl"
    if not IN_COLAB:
        local = REPO_ROOT / "data" / f"codesearchnet_{language}" / fname
        if local.is_file():
            return local
        raise FileNotFoundError(f"Local file not found: {local}")

    candidates = [
        CANONICAL_INPUT,
        DRIVE_ROOT / "jsonl" / fname,
        DRIVE_ROOT / fname,
        Path("/content/drive/MyDrive") / fname,
    ]
    for p in candidates:
        if p.is_file():
            return p

    found = sorted(Path("/content/drive/MyDrive").rglob(fname))
    if found:
        print(f"Found on Drive: {found[0]}")
        return found[0]

    raise FileNotFoundError(
        f"{fname} not found after Drive mount. "
        f"Upload to {DRIVE_ROOT / 'jsonl' / fname} (recommended), then re-run this cell."
    )


INPUT_JSONL = resolve_input_jsonl(LANGUAGE)
require_language_support(LANGUAGE)
print(f"LANGUAGE: {LANGUAGE}")
print(f"java scripts: {_has_java_scripts(REPO_ROOT)}")
print(f"INPUT_JSONL: {INPUT_JSONL}  ({INPUT_JSONL.stat().st_size / 1e6:.0f} MB)")
print(f"MANIFEST: {MANIFEST}")
print(f"NPZ_DIR: {NPZ_DIR}")

## Sanity checks (GPU)

In [13]:
device = "cuda" if torch.cuda.is_available() else "cpu"
require_language_support(LANGUAGE)
for args in [
    ["scripts/qwen_inference.py", "verify", "--device", device],
    ["scripts/token_alignment.py", "verify"],
    ["scripts/variable_occurrences.py", "verify", "--language", LANGUAGE],
    ["scripts/activation_pipeline.py", "verify", "--device", device, "--language", LANGUAGE],
]:
    cmd = [sys.executable, *args]
    print("$", " ".join(cmd))
    proc = subprocess.run(cmd, cwd=REPO_ROOT, capture_output=True, text=True)
    if proc.stdout:
        print(proc.stdout.rstrip())
    if proc.returncode != 0:
        if proc.stderr:
            print(proc.stderr.rstrip())
        raise subprocess.CalledProcessError(proc.returncode, cmd)

## Boolean-flag occurrences (optional, CPU)

`activation_pipeline` finds occurrences inline during GPU extraction. Set `EXTRACT_OCCURRENCES = True` in Configuration to also write a standalone JSONL for inspection.

In [14]:
if EXTRACT_OCCURRENCES:
    occ_args = [
        "scripts/variable_occurrences.py", "extract",
        "--language", LANGUAGE,
        "--input", str(INPUT_JSONL),
        "--output", str(OCCURRENCES_JSONL),
        "--model-id", MODEL_ID,
        "--max-rows", str(MAX_ROWS),
    ]
    print("$", " ".join([sys.executable, *occ_args]))
    subprocess.check_call([sys.executable, *occ_args], cwd=REPO_ROOT)
    n_lines = sum(1 for ln in OCCURRENCES_JSONL.read_text().splitlines() if ln.strip())
    print(f"Wrote {n_lines} occurrence lines -> {OCCURRENCES_JSONL}")
else:
    print("Skipping standalone occurrence extract (set EXTRACT_OCCURRENCES=True to enable).")

## Activation extraction (GPU)

### Emergency stop (when Interrupt does not work)
1. **Runtime → Restart runtime** (kills GPU immediately; progress on Drive is kept)
2. Or in Drive file browser: create empty file  
   `My Drive/mech-interp/activations_java/STOP_EXTRACTION`  
   (pipeline checks this every row on the **next** pull)

Probe mode auto-stops at `TARGET_NPZ` / `TARGET_COMPLETE_ROWS`. Default: **1 chunk per run** (`MAX_CHUNKS=1`) — re-run the cell to continue with `RESUME=True`.

In [15]:
import json
from tqdm.auto import tqdm

from activation_pipeline import completed_source_rows

def _count_complete_rows() -> int:
    return len(completed_source_rows(MANIFEST, NPZ_DIR))

complete_rows = _count_complete_rows()
lines = MANIFEST.read_text(encoding="utf-8").splitlines() if MANIFEST.is_file() else []
with_npz = sum(1 for ln in lines if ln.strip() and json.loads(ln).get("activation_path"))
marked_complete = sum(1 for ln in lines if ln.strip() and json.loads(ln).get("row_status") == "complete")
parse_errors = sum(1 for ln in lines if ln.strip() and json.loads(ln).get("parse_error"))
no_occ = sum(1 for ln in lines if ln.strip() and json.loads(ln).get("occurrence_count") == 0)
npz_count = len(list(NPZ_DIR.glob("*.npz")))

print(f"manifest exists: {MANIFEST.is_file()}")
print(f"manifest lines: {len(lines)}")
print(f"rows complete (resume): {complete_rows}  (row_status lines: {marked_complete})")
print(f"rows with npz lines:    {with_npz}")
print(f"parse errors:           {parse_errors}")
print(f"no occurrences:         {no_occ}")
print(f"npz files on disk:      {npz_count}")

if not INPUT_JSONL.is_file():
    raise FileNotFoundError(
        f"Upload canonical JSONL to {INPUT_JSONL} on Google Drive first."
    )


def _count_input_rows(path: Path) -> int:
    n = 0
    with path.open(encoding="utf-8") as f:
        for line in tqdm(f, desc="Counting input JSONL lines", unit="line"):
            if line.strip():
                n += 1
    return n


if TOTAL_INPUT_ROWS is None:
    TOTAL_INPUT_ROWS = _count_input_rows(INPUT_JSONL)
    print(f"TOTAL_INPUT_ROWS={TOTAL_INPUT_ROWS}")

# Clear stop flag from a previous run
if STOP_FILE.is_file():
    STOP_FILE.unlink()
    print(f"Removed old stop file: {STOP_FILE}")

EXTRACTION_TARGET = (
    TARGET_COMPLETE_ROWS if EXTRACTION_MODE == "probe" else TOTAL_INPUT_ROWS
)

if EXTRACTION_MODE == "probe" and npz_count >= TARGET_NPZ:
    print(
        f"SKIP extraction: already have {npz_count} npz (target {TARGET_NPZ}). "
        f"Go to Probing — PROBE_MAX_SAMPLES={PROBE_MAX_SAMPLES}."
    )
elif EXTRACTION_MODE == "probe" and complete_rows >= TARGET_COMPLETE_ROWS:
    print(
        f"SKIP extraction: already have {complete_rows} complete input rows "
        f"(target {TARGET_COMPLETE_ROWS}). Go to Probing."
    )
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    overall = tqdm(
        total=EXTRACTION_TARGET,
        initial=min(complete_rows, EXTRACTION_TARGET),
        desc=f"extraction ({EXTRACTION_MODE})",
        unit="row",
        dynamic_ncols=True,
    )
    chunk = 0
    after_complete = complete_rows
    while True:
        chunk += 1
        if MAX_CHUNKS is not None and chunk > int(MAX_CHUNKS):
            print(f"Stopped after MAX_CHUNKS={MAX_CHUNKS}")
            break

        before_complete = _count_complete_rows()
        npz_before = len(list(NPZ_DIR.glob("*.npz")))
        extract_args = [
            "scripts/activation_pipeline.py", "extract",
            "--language", LANGUAGE,
            "--input", str(INPUT_JSONL),
            "--manifest", str(MANIFEST),
            "--tensor-dir", str(NPZ_DIR),
            "--model-id", MODEL_ID,
            "--device", device,
            "--dtype", "bf16",
            "--max-rows", str(CHUNK_ROWS),
        ]
        if RESUME:
            extract_args.append("--resume")
        extract_args.extend(["--stop-file", str(STOP_FILE)])

        print(
            f"\n=== chunk {chunk} | goal {EXTRACTION_TARGET} rows / {TARGET_NPZ} npz "
            f"| now {before_complete} rows, {npz_before} npz ==="
        )
        cmd = [sys.executable, *extract_args]
        print("$", " ".join(cmd))
        proc = subprocess.run(cmd, cwd=REPO_ROOT)
        if proc.returncode != 0:
            raise subprocess.CalledProcessError(proc.returncode, cmd)

        after_complete = _count_complete_rows()
        npz_after = len(list(NPZ_DIR.glob("*.npz")))
        overall.update(max(0, after_complete - before_complete))
        overall.set_postfix(npz=npz_after, chunk=chunk)

        if after_complete == before_complete:
            print("No new rows this chunk — extraction caught up.")
            break
        if EXTRACTION_MODE == "probe" and after_complete >= TARGET_COMPLETE_ROWS:
            print(f"Probe row goal reached ({after_complete} >= {TARGET_COMPLETE_ROWS}).")
            break
        if EXTRACTION_MODE == "probe" and npz_after >= TARGET_NPZ:
            print(f"Probe npz goal reached ({npz_after} >= {TARGET_NPZ}).")
            break

    overall.close()
    print(
        f"Done: {after_complete}/{EXTRACTION_TARGET} target rows, "
        f"{len(list(NPZ_DIR.glob('*.npz')))} npz — ready for probing"
    )

In [16]:
import glob

lines = MANIFEST.read_text(encoding="utf-8").splitlines() if MANIFEST.is_file() else []
with_npz = sum(1 for ln in lines if ln.strip() and json.loads(ln).get("activation_path"))
print(f"manifest lines: {len(lines)}")
print(f"rows with .npz:  {with_npz}")
print(f"npz files:       {len(glob.glob(str(NPZ_DIR / '*.npz')))}")

## Probing (CPU — no model load)

Trains a logistic regression probe per layer on saved activations.

**Target:** `occurrence_type` (e.g. `conditional_use` vs `assignment` vs `return_use`).
This uses real pipeline data; for multi-role probing you will need iterator/indexing labels later.

## Fast probing cache

Copy up to `PROBE_MAX_SAMPLES` `.npz` files to local disk (Drive random reads are slow). Uses the same limit as the probe trainer below.

In [ ]:
from pathlib import Path
import shutil
import zipfile
from tqdm.auto import tqdm

LOCAL_NPZ_DIR = Path('/content/npz_cache')
LOCAL_NPZ_DIR.mkdir(parents=True, exist_ok=True)

COPY_LIMIT = PROBE_MAX_SAMPLES
RECOPY_CORRUPT = True  # validate existing cache files and recopy broken ones

drive_files = sorted(NPZ_DIR.glob('*.npz'))
if COPY_LIMIT is not None:
    drive_files = drive_files[:int(COPY_LIMIT)]


def _valid_npz(path: Path) -> bool:
    try:
        with np.load(path) as data:
            return 'mean' in data and 'first' in data and 'last' in data
    except (EOFError, zipfile.BadZipFile, ValueError, OSError):
        return False


copied = 0
skipped = 0
repaired = 0
for src in tqdm(drive_files, desc='Copying npz to /content', unit='file'):
    dst = LOCAL_NPZ_DIR / src.name
    if dst.exists():
        if RECOPY_CORRUPT and not _valid_npz(dst):
            tmp = dst.with_suffix('.tmp')
            shutil.copy2(src, tmp)
            tmp.replace(dst)
            repaired += 1
        else:
            skipped += 1
        continue

    tmp = dst.with_suffix('.tmp')
    shutil.copy2(src, tmp)
    tmp.replace(dst)
    copied += 1

print(f'Drive npz considered: {len(drive_files)}')
print(f'Copied: {copied}, repaired(corrupt): {repaired}, skipped(existing): {skipped}')
print(f'Local npz now: {len(list(LOCAL_NPZ_DIR.glob("*.npz")))} at {LOCAL_NPZ_DIR}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import zipfile
from collections import Counter
from tqdm.auto import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

MIN_CLASS_COUNT = 20
RANDOM_STATE = 42
FAST_MAX_SAMPLES = PROBE_MAX_SAMPLES

local_npz_dir = Path('/content/npz_cache')
if not local_npz_dir.is_dir() or not any(local_npz_dir.glob('*.npz')):
    raise RuntimeError('No local cache found. Run the cache cell first.')

manifest_lines = MANIFEST.read_text(encoding='utf-8').splitlines()
records = []
bad_npz = 0
missing_npz = 0
for ln in tqdm(manifest_lines, desc='Loading cached activations', unit='line'):
    if not ln.strip():
        continue
    rec = json.loads(ln)
    rel = rec.get('activation_path')
    occ_type = rec.get('occurrence_type')
    if not rel or not occ_type:
        continue
    npz_path = local_npz_dir / Path(rel).name
    if not npz_path.is_file():
        missing_npz += 1
        continue
    try:
        with np.load(npz_path) as data:
            if POOLING not in data:
                continue
            records.append({'y': occ_type, 'X': data[POOLING].astype(np.float32)})
    except (EOFError, zipfile.BadZipFile, ValueError, OSError):
        bad_npz += 1
        continue

print(f'Loaded records: {len(records)} | missing npz: {missing_npz} | corrupt npz skipped: {bad_npz}')
if not records:
    raise RuntimeError('No usable activation records found in local cache.')

if FAST_MAX_SAMPLES is not None and len(records) > FAST_MAX_SAMPLES:
    rng = np.random.default_rng(RANDOM_STATE)
    idx = rng.choice(len(records), size=int(FAST_MAX_SAMPLES), replace=False)
    records = [records[int(i)] for i in idx]

counts = Counter(r['y'] for r in records)
keep = {k for k, v in counts.items() if v >= MIN_CLASS_COUNT}
records = [r for r in records if r['y'] in keep]
print(f'Probe samples: {len(records)}  classes: {dict(Counter(r["y"] for r in records))}')

le = LabelEncoder()
y = le.fit_transform([r['y'] for r in records])
n_layers = records[0]['X'].shape[0]

layer_accs, layer_f1s = [], []
for layer in tqdm(range(n_layers), desc='Training layer probes', unit='layer'):
    X = np.stack([r['X'][layer] for r in records], axis=0)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )
    clf = LogisticRegression(max_iter=1200, solver='saga', random_state=RANDOM_STATE)
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    layer_accs.append(accuracy_score(y_test, pred))
    layer_f1s.append(f1_score(y_test, pred, average='macro'))

best = int(np.argmax(layer_f1s))
print(f'Best layer (macro F1): {best}  acc={layer_accs[best]:.3f}  f1={layer_f1s[best]:.3f}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(layer_accs, label='accuracy')
ax.plot(layer_f1s, label='macro F1')
ax.axvline(best, color='gray', ls='--', alpha=0.6)
ax.set_xlabel('layer')
ax.set_ylabel('score')
ax.set_title(f'{LANGUAGE} occurrence_type probe ({POOLING} pooling, n={len(records)})')
ax.legend()
plt.tight_layout()
plt.show()